# **2. Visitor Type Data**

### 0.1 Mobility Data (type = 'stop')

In [ ]:
# 1. Load Canary Wharf Mobility Data
import pandas as pd
import numpy as np

# Load the CSV file
csv_file_path = '/content/drive/MyDrive/CUSP-GX 7103 7113 CAPSTONE/CAPSTONE WORK PROGRESS/Data/Sample_CanaryWharf/canarywharf_r10_2025-03_04.csv'
canarywharf_df = pd.read_csv(csv_file_path)

print(f"Loaded CSV file: {csv_file_path}")
display(canarywharf_df.head())

Loaded CSV file: /content/drive/MyDrive/CUSP-GX 7103 7113 CAPSTONE/CAPSTONE WORK PROGRESS/Data/Sample_CanaryWharf/canarywharf_r10_2025-03_04.csv


,id,lat,lon,ts,date,gridcell,type
0,3fe1f077e29799ec1b28a80d42304b8f,51.507560,-0.024580,1743382001,2025-03-31,8a194ad266d7fff,stop
1,5549c0383dd1440e3288f4b1c18f750d,51.507626,-0.025254,1743365709,2025-03-30,8a194ad266d7fff,pedestrian
2,cbd87fa1a40281775f4655f875103f54,51.507140,-0.024240,1743082018,2025-03-27,8a194ad266d7fff,pedestrian
3,c1a1163c2b9defb34b767b4e440916a8,51.507236,-0.025280,1742873344,2025-03-25,8a194ad266d7fff,stop
4,a18d4344aada48028aa2a4330c1ae0a8,51.507580,-0.024490,1743019306,2025-03-26,8a194ad266d7fff,pedestrian


In [ ]:
stop_df = canarywharf_df[canarywharf_df['type'] == 'stop'].copy()
stop_df.shape

(1876684, 7)

### 0.2 POI Data

#### 0.2.1 poi aggregation -> super poi
Identify 140 building with as "owned" polygon

In [ ]:
# 2. Load Canary Wharf POI Data (Safegraph)
import geopandas as gpd

geojason_file_path = '/content/drive/MyDrive/CUSP-GX 7103 7113 CAPSTONE/CAPSTONE WORK PROGRESS/Data/Sample_CanaryWharf/poi_within_canary_wharf.geojson'
poi_gdf = gpd.read_file(geojason_file_path)

print(f"Loaded GeoJSON file from: {geojason_file_path}")
display(poi_gdf.head())

Loaded GeoJSON file from: /content/drive/MyDrive/CUSP-GX 7103 7113 CAPSTONE/CAPSTONE WORK PROGRESS/Data/Sample_CanaryWharf/poi_within_canary_wharf.geojson


,BRANDS,CATEGORY_TAGS,CITY,CLOSED_ON,DOMAINS,ENCLOSED,GEOMETRY_TYPE,INCLUDES_PARKING_LOT,ISO_COUNTRY_CODE,IS_SYNTHETIC,...,entity,name,dataset,typology,reference,prefix,organisation-entity,quality,description,geometry
0,[],[],London,NaT,[htl.london],False,POLYGON,False,GB,False,...,7010004226,Canary Wharf,article-4-direction-area,geography,LBTH_Art4_10,article-4-direction-area,350,authoritative,The council has confirmed an Article 4 directi...,POINT (-0.01482 51.50373)
1,[{'safegraph_brand_id': 'SG_BRAND_b9a4e4545918...,[ATMs],Canary Wharf,NaT,[hsbc.com],None,POINT,None,GB,None,...,7010004226,Canary Wharf,article-4-direction-area,geography,LBTH_Art4_10,article-4-direction-area,350,authoritative,The council has confirmed an Article 4 directi...,POINT (-0.01744 51.50444)
2,[],[Shopping Mall],London,NaT,[],False,POLYGON,False,GB,False,...,7010004226,Canary Wharf,article-4-direction-area,geography,LBTH_Art4_10,article-4-direction-area,350,authoritative,The council has confirmed an Article 4 directi...,POINT (-0.01872 51.50253)
3,[],[],London,NaT,[entralon.com],False,POLYGON,False,GB,False,...,7010004226,Canary Wharf,article-4-direction-area,geography,LBTH_Art4_10,article-4-direction-area,350,authoritative,The council has confirmed an Article 4 directi...,POINT (-0.01959 51.50508)
4,[],[],London,NaT,[currencyglobal.com],False,POLYGON,False,GB,False,...,7010004226,Canary Wharf,article-4-direction-area,geography,LBTH_Art4_10,article-4-direction-area,350,authoritative,The council has confirmed an Article 4 directi...,POINT (-0.01486 51.50371)


In [ ]:
import geopandas as gpd
import pandas as pd
from shapely import wkt
import numpy as np

# --- 1. 数据清洗与预处理 ---
print("正在初始化数据并恢复几何对象...")

# 确保 geometry 是从 POLYGON_WKT 恢复的
if 'POLYGON_WKT' in poi_gdf.columns:
    poi_polygons = poi_gdf['POLYGON_WKT'].apply(lambda x: wkt.loads(x) if pd.notnull(x) else None)
    poi_clean = gpd.GeoDataFrame(poi_gdf.drop(columns='geometry', errors='ignore'),
                                 geometry=poi_polygons, crs="EPSG:4326")
else:
    poi_clean = poi_gdf.copy()

# 转换为英国投影坐标系 (BNG) 以进行精确面积计算
poi_clean = poi_clean.to_crs(epsg=27700)
poi_clean = poi_clean[poi_clean.geometry.notnull() & (~poi_clean.geometry.is_empty)]

# --- 2. 核心逻辑：定义“超级 ID” (Super ID Assignment) ---
# 策略：如果一个 POI 有 PARENT_PLACEKEY，它就属于那个父级建筑；如果没有，它自己就是 ID。
print("正在根据建筑层级定义 Super ID...")
poi_clean['SUPER_ID'] = poi_clean['PARENT_PLACEKEY'].fillna(poi_clean['PLACEKEY'])

# --- 3. 第一次聚合：基于语义 ID 的合并 ---
def aggregate_by_id(group):
    # 几何合并：使用新版 union_all()
    merged_geom = group.geometry.union_all()

    # 找到该组中面积最大的 POI 作为“代表” (Building Anchor)
    # 优先使用原始数据提供的 WKT_AREA_SQ_METERS
    host_row = group.loc[group['WKT_AREA_SQ_METERS'].idxmax()] if 'WKT_AREA_SQ_METERS' in group.columns else group.iloc[0]

    # 统计信息
    names = group['LOCATION_NAME'].unique().tolist()
    cats = group['TOP_CATEGORY'].value_counts().index.tolist()

    return pd.Series({
        'geometry': merged_geom,
        'building_name': host_row['LOCATION_NAME'],
        'main_category': cats[0] if cats else 'N/A',
        'all_categories': cats[:5],
        'shop_list': names,
        'shop_count': len(names),
        'total_area': merged_geom.area
    })

print("执行第一轮语义聚合...")
building_nodes = poi_clean.groupby('SUPER_ID').apply(aggregate_by_id, include_groups=False).reset_index()
building_nodes = gpd.GeoDataFrame(building_nodes, geometry='geometry', crs="EPSG:27700")

# --- 4. 第二次聚合：解决空间“幽灵重叠” (Spatial Deduplication) ---
# 逻辑：即便 PARENT_PLACEKEY 不同，但如果两个多边形重叠率极高 (>80%)，则视为同一实体
print("执行第二轮空间去重（解决块 A/B 这种几何重叠问题）...")

# 自交检测
overlaps = gpd.sjoin(building_nodes, building_nodes[['geometry']], how='inner', predicate='intersects')
overlaps = overlaps[overlaps.index != overlaps.index_right] # 排除自身

def get_iou(idx_left, idx_right):
    g1 = building_nodes.loc[idx_left].geometry
    g2 = building_nodes.loc[idx_right].geometry
    inter = g1.intersection(g2).area
    return inter / min(g1.area, g2.area)

# 计算重叠率并过滤
if not overlaps.empty:
    overlaps['iou'] = overlaps.apply(lambda x: get_iou(x.name, x.index_right), axis=1)
    to_merge = overlaps[overlaps['iou'] > 0.8]

    # 简单的消冗逻辑：保留面积大的 ID
    drop_indices = set()
    for idx, row in to_merge.iterrows():
        if idx not in drop_indices and row.index_right not in drop_indices:
            # 比较面积，丢弃小的
            if building_nodes.loc[idx].total_area >= building_nodes.loc[row.index_right].total_area:
                drop_indices.add(row.index_right)
            else:
                drop_indices.add(idx)

    final_nodes = building_nodes.drop(index=list(drop_indices)).reset_index(drop=True)
else:
    final_nodes = building_nodes

# --- 5. 最终清理与输出 ---
final_nodes['geometry'] = final_nodes.geometry.simplify(0.2) # 几何微简化，提升渲染性能
final_nodes = final_nodes[final_nodes.geometry.is_valid]

print(f"✅ 聚合完成！")
print(f"原始 POI 数量: {len(poi_gdf)}")
print(f"语义聚合后建筑数: {len(building_nodes)}")
print(f"空间去重后最终节点数: {len(final_nodes)}")

# 保存结果
# final_nodes.to_crs(epsg=4326).to_file("Canary_Wharf_SuperNodes.geojson", driver='GeoJSON')

正在初始化数据并恢复几何对象...
正在根据建筑层级定义 Super ID...
执行第一轮语义聚合...
执行第二轮空间去重（解决块 A/B 这种几何重叠问题）...
✅ 聚合完成！
原始 POI 数量: 1377
语义聚合后建筑数: 767
空间去重后最终节点数: 139


In [ ]:
import folium
from folium.plugins import MarkerCluster

# 1. 坐标转换回 WGS84 以适配 Folium
final_map_4326 = final_nodes.to_crs(epsg=4326)

# 2. 初始化地图 (Canary Wharf 中心点)
m = folium.Map(location=[51.5048, -0.0210], zoom_start=16, tiles='CartoDB positron')

# 3. 添加建筑轮廓
for _, row in final_map_4326.iterrows():
    # 根据商店数量决定填充颜色 (越多越红)
    color = '#e67e22' if row['shop_count'] < 5 else '#e74c3c' if row['shop_count'] < 20 else '#c0392b'

    # 格式化 Shop List HTML
    shops = row['shop_list'][:20] # 仅显示前20个防止弹窗过长
    shop_html = "".join([f"<li>{s}</li>" for s in shops])
    if row['shop_count'] > 20:
        shop_html += f"<li>...以及其他 {row['shop_count']-20} 家</li>"

    popup_text = f"""
    <div style="width:250px; font-family: sans-serif;">
        <h4 style="margin-bottom:5px;">{row['building_name']}</h4>
        <p><b>主营属性:</b> {row['main_category']}</p>
        <p><b>店铺总数:</b> {row['shop_count']}</p>
        <hr>
        <ul style="max-height:150px; overflow-y:auto; padding-left:15px;">
            {shop_html}
        </ul>
    </div>
    """

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': '#34495e',
            'weight': 1.5,
            'fillOpacity': 0.7
        },
        tooltip=f"{row['building_name']} ({row['shop_count']} shops)",
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# 4. 保存并显示
# m.save("Canary_Wharf_Building_Intelligence.html")
# print("🔥 交互地图已生成！请打开 Canary_Wharf_Building_Intelligence.html 查看。")

In [ ]:
m

In [ ]:
# 修正 KeyError 的安全提取函数
def get_safe_col(df, base_name):
    for col in [base_name, f"{base_name}_left", f"{base_name}_right"]:
        if col in df.columns:
            return col
    return None

# 定位目标行
target_a_ids = block_a_sample['SUPER_ID'].unique()
target_b_ids = block_b_sample['SUPER_ID'].unique()

print(f"检测到块 A 有 {len(target_a_ids)} 个重复 ID, 块 B 有 {len(target_b_ids)} 个重复 ID")

# 从 final_merge 提取详细信息
relevant_data = final_merge[final_merge['SUPER_ID'].isin(list(target_a_ids) + list(target_b_ids))].copy()

# 提取 Placekey 前缀
relevant_data['pk_prefix'] = relevant_data['PLACEKEY_left'].str.split('@').str[0]

# 获取正确的类别列名
cat_col = get_safe_col(relevant_data, 'TOP_CATEGORY')

for tid in list(target_a_ids[:1]) + list(target_b_ids[:1]): # 每组只看一个代表
    group = relevant_data[relevant_data['SUPER_ID'] == tid]
    print(f"\n--- 分析 SUPER_ID: {tid} ---")
    print(f"主要地址前缀 (Placekey Prefix): {group['pk_prefix'].value_counts().head(3)}")

    if cat_col:
        print(f"前 5 大类别分布:\n{group[cat_col].value_counts().head(5)}")

    # 检查 Parent Placekey
    parent_col = get_safe_col(group, 'PARENT_PLACEKEY')
    if parent_col:
        parents = group[parent_col].dropna().unique()
        print(f"父级建筑 ID (Parent Placekeys): {parents}")

检测到块 A 有 4 个重复 ID, 块 B 有 3 个重复 ID

--- 分析 SUPER_ID: zzy-229@4hh-ztp-59f ---
主要地址前缀 (Placekey Prefix): pk_prefix
zzy-26h    61
zzy-26q    61
zzy-26g    61
Name: count, dtype: int64
前 5 大类别分布:
TOP_CATEGORY
Restaurants and Other Eating Places             383
Personal Care Services                          137
Clothing Stores                                 122
Shoe Stores                                      61
Office Supplies, Stationery, and Gift Stores     61
Name: count, dtype: int64
父级建筑 ID (Parent Placekeys): []

--- 分析 SUPER_ID: zzy-2ft@4hh-ztn-s3q ---
主要地址前缀 (Placekey Prefix): pk_prefix
zzy-2b2    176
zzy-2fv    176
zzy-2fb    176
Name: count, dtype: int64
前 5 大类别分布:
TOP_CATEGORY
Restaurants and Other Eating Places                               704
Accounting, Tax Preparation, Bookkeeping, and Payroll Services    704
Jewelry, Luggage, and Leather Goods Stores                        528
Other Financial Investment Activities                             528
Clothing Stores           

In [ ]:
print('Unique POLYGON_CLASS categories:')
print(poi_gdf['POLYGON_CLASS'].unique())

owned_count = (poi_gdf['POLYGON_CLASS'] == 'OWNED_POLYGON').sum()
print(f"\nNumber of POIs with POLYGON_CLASS == 'OWNED_POLYGON': {owned_count}")

Unique POLYGON_CLASS categories:
['SHARED_POLYGON' None 'OWNED_POLYGON']

Number of POIs with POLYGON_CLASS == 'OWNED_POLYGON': 140


In [ ]:
owned_polygon_ids = poi_gdf[poi_gdf['POLYGON_CLASS'] == 'OWNED_POLYGON']['PLACEKEY']
print("IDs of POIs with 'OWNED_POLYGON':")
for pid in owned_polygon_ids:
    print(pid)

In [ ]:
owned_polygons_df = poi_gdf[poi_gdf['POLYGON_CLASS'] == 'OWNED_POLYGON']

print('BRANDS (for OWNED_POLYGON):')
print(owned_polygons_df['BRANDS'].head())
print('\nCATEGORY_TAGS (for OWNED_POLYGON):')
print(owned_polygons_df['CATEGORY_TAGS'].head())
print('\nLOCATION_NAME (for OWNED_POLYGON):')
print(owned_polygons_df['LOCATION_NAME'].head())

BRANDS (for OWNED_POLYGON):
2     []
55    []
56    []
58    []
59    []
Name: BRANDS, dtype: object

CATEGORY_TAGS (for OWNED_POLYGON):
2                                       [Shopping Mall]
55                                                   []
56                                  [Corporate Offices]
58    [Bar or Pub, Beer, Casual Dining, Cocktail Lou...
59                                         [Industrial]
Name: CATEGORY_TAGS, dtype: object

DOMAINS (for OWNED_POLYGON):
2                         []
55       [ivoryresearch.com]
56                        []
58    [drakeandmorgan.co.uk]
59                        []
Name: DOMAINS, dtype: object

LOCATION_NAME (for OWNED_POLYGON):
2                               London Bank Street
55    Mehrdad Tahira Research and Training Network
56                            W.W. Grainger Office
58                                The Sipping Room
59                           Elstree Data Center 4
Name: LOCATION_NAME, dtype: object


In [ ]:
import pandas as pd

# --- 1. 筛选原始数据中的 Owned Polygons ---
owned_buildings = poi_gdf[poi_gdf['POLYGON_CLASS'] == 'OWNED_POLYGON'].copy()

# --- 2. 构建“建筑身份”识别逻辑 ---
def identify_building(row):
    name = str(row['LOCATION_NAME'])
    brand = str(row['BRANDS']) if pd.notnull(row['BRANDS']) else ""
    address = str(row['STREET_ADDRESS']) if pd.notnull(row['STREET_ADDRESS']) else ""

    # 如果品牌名和大楼名不一致，合并显示（例如：HSBC - 8 Canada Square）
    if brand and brand.lower() not in name.lower():
        return f"{brand} ({name}) | {address}"
    return f"{name} | {address}"

owned_buildings['building_identity'] = owned_buildings.apply(identify_building, axis=1)

# --- 3. 按照规模（面积）排序，看看谁是大家伙 ---
# 假设 WKT_AREA_SQ_METERS 存在
if 'WKT_AREA_SQ_METERS' in owned_buildings.columns:
    owned_buildings = owned_buildings.sort_values('WKT_AREA_SQ_METERS', ascending=False)

# --- 4. 打印结果 ---
print(f"找到 {len(owned_buildings)} 个独立拥有的建筑实体：\n")
print(owned_buildings[['building_identity', 'TOP_CATEGORY', 'WKT_AREA_SQ_METERS']].to_string(index=False))

# --- 5. 快速统计分类（反向推导 Landuse） ---
print("\n--- 建筑功能分布推测 ---")
print(owned_buildings['TOP_CATEGORY'].value_counts().head(15))

找到 140 个独立拥有的建筑实体：

                                                                                                                                                                        building_identity                                                   TOP_CATEGORY  WKT_AREA_SQ_METERS
                                                                                                                                                     London Trafalgar Way | Trafalgar Way                                         Lessors of Real Estate             10985.0
                                                                                                                       Elstree Data Center 4 | Hertsmere Road Beluga Cafe 1 The Warehouse                 Data Processing, Hosting, and Related Services             10190.0
                                                                                                                                                    TheForexCentre Ltd | 25 C

/tmp/ipykernel_8424/1929965350.py:9: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  brand = str(row['BRANDS']) if pd.notnull(row['BRANDS']) else ""


In [ ]:
# 打印这 140 个节点的信息
print(final_nodes[['building_name', 'shop_count', 'main_category']].sort_values('shop_count', ascending=False).to_string())

                                                         building_name  shop_count                                                   main_category
129                                                       YR Mortgages          62  Accounting, Tax Preparation, Bookkeeping, and Payroll Services
136                                            CCT Venues-Canary Wharf          61  Accounting, Tax Preparation, Bookkeeping, and Payroll Services
122                                                        HTL Support          53  Accounting, Tax Preparation, Bookkeeping, and Payroll Services
120                                                       GI Cognition          45  Accounting, Tax Preparation, Bookkeeping, and Payroll Services
27                                        Canary Wharf Shopping Centre          30                             Restaurants and Other Eating Places
91                                                        Franco Manca          26                             Restaur

#### 0.2.2 landuse, and semantic extraction

In [ ]:
import folium
import pandas as pd

# --- 1. Define Category Colors (Architectural Palette) ---
# Blue for office, red for retail, purple for mixed-use, yellow for infrastructure, grey for other commercial
color_map = {
    "High-value Office (HQ)": "#3498db",            # Blue
    "Retail Hub / Shopping Mall": "#e74c3c",       # Red
    "Vertical Mixed-Use": "#9b59b6",               # Purple
    "Educational / Social Infrastructure": "#f1c40f", # Yellow
    "Commercial - Other": "#95a5a6"                # Grey
}

# --- 2. Rerun Logic: Infer Landuse based on POI Proportions ---
def infer_landuse_v3(row):
    cats = str(row['all_categories']).lower()
    name = str(row['building_name']).lower()
    count = row['shop_count']

    # Priority identification: Office/Headquarters (for developer tags like 'Lessors of Real Estate')
    if 'lessors of real estate' in cats or 'management of companies' in cats or 'financial' in cats:
        if count < 15: return "High-value Office (HQ)"
        else: return "Vertical Mixed-Use"

    # Identification: Retail Hub (e.g., Jubilee Place)
    if 'restaurants' in cats or 'clothing stores' in cats:
        if count > 20: return "Retail Hub / Shopping Mall"
        return "Vertical Mixed-Use" if 'financial' in cats else "Retail Hub / Shopping Mall"

    # Identification: Social Infrastructure
    if 'school' in cats or 'day care' in cats or 'museum' in cats:
        return "Educational / Social Infrastructure"

    return "Commercial - Other"

# Apply inference logic
final_nodes['inferred_landuse'] = final_nodes.apply(infer_landuse_v3, axis=1)

# --- 3. Generate Map ---
final_map_4326 = final_nodes.to_crs(epsg=4326)
m = folium.Map(location=[51.5048, -0.0210], zoom_start=16, tiles='CartoDB positron')

for _, row in final_map_4326.iterrows():
    landuse = row['inferred_landuse']
    fill_color = color_map.get(landuse, "#95a5a6")

    # Prepare popup HTML
    shop_list_html = "".join([f"<li>{str(s)}</li>" for s in row['shop_list'][:25]])
    if row['shop_count'] > 25:
        shop_list_html += f"<li>...and {row['shop_count']-25} more units</li>"

    popup_content = f"""
    <div style="width:280px; font-family: Arial, sans-serif;">
        <h4 style="color:{fill_color}; margin-bottom:5px;">{row['building_name']}</h4>
        <p style="margin:2px 0; font-size:13px;"><b>Inferred Type:</b> {landuse}</p>
        <p style="margin:2px 0; font-size:13px;"><b>Total Units:</b> {row['shop_count']}</p>
        <hr style="border:0.5px solid #eee;">
        <p style="font-size:12px; font-weight:bold; margin-bottom:5px;">Internal POIs (partial):</p>
        <ul style="max-height:120px; overflow-y:auto; font-size:11px; padding-left:18px; color:#555;">
            {shop_list_html}
        </ul>
    </div>
    """

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, fc=fill_color: {
            'fillColor': fc,
            'color': '#2c3e50',
            'weight': 1,
            'fillOpacity': 0.7
        },
        highlight_function=lambda x: {'weight': 3, 'fillOpacity': 0.9},
        tooltip=f"{row['building_name']} ({landuse})",
        popup=folium.Popup(popup_content, max_width=300)
    ).add_to(m)

# --- 4. Add Pure HTML Legend ---
legend_html = f'''
     <div style="position: fixed; bottom: 50px; left: 50px; width: 230px; height: 160px;
     background-color: white; border:2px solid grey; z-index:9999; font-size:12px;
     padding: 10px; border-radius: 5px;">
     <b>Building Landuse (Inferred)</b><br>
     &nbsp; <span style="color:#3498db; font-size:20px;">■</span>&nbsp; Office (High-value HQ)<br>
     &nbsp; <span style="color:#e74c3c; font-size:20px;">■</span>&nbsp; Retail Hub / Shopping Mall<br>
     &nbsp; <span style="color:#9b59b6; font-size:20px;">■</span>&nbsp; Vertical Mixed-Use<br>
     &nbsp; <span style="color:#f1c40f; font-size:20px;">■</span>&nbsp; Educational / Social<br>
     &nbsp; <span style="color:#95a5a6; font-size:20px;">■</span>&nbsp; Other / General<br>
     </div>
     '''
m.get_root().html.add_child(folium.Element(legend_html))

m.save("Canary_Wharf_Functional_Intelligence.html")
print("🔥 Fix completed! Map saved as Canary_Wharf_Functional_Intelligence.html")

🔥 Fix completed! Map saved as Canary_Wharf_Functional_Intelligence.html


In [ ]:
m